# Dropbox

Description:
- Interact with data in dropbox.com. Includes the Official Api Spec Language for Dropbox (Stone).

Prerequisites:
- See [authentication.ipynb] 

Resources:
- [GitHub] https://github.com/dropbox/dropbox-sdk-python/tree/main
- [Examples] https://github.com/dropbox/dropbox-sdk-python/tree/main/example
- [Docs] https://dropbox-sdk-python.readthedocs.io/en/latest/
- [Developer Page] https://www.dropbox.com/developers/documentation/python#tutorial
- [GitHub] https://github.com/dropbox/stone?tab=readme-ov-file
- [More Docs] https://github.com/dropbox/stone/tree/main/docs
- [Docs] https://github.com/dropbox/stone/blob/main/docs/lang_ref.rst#documentation

## Environment variables

This notebook reads its settings from environment variables. Set them before starting Jupyter:

- `DROPBOX_ACCESS_TOKEN`: a Dropbox OAuth2 access token (see `authentication.ipynb`)

## Example 1

Create a client from an access token, then upload a local file, replacing any existing copy

In [ ]:
import os
import tempfile
from pathlib import Path

import dropbox
from dropbox.exceptions import ApiError, AuthError
from dropbox.files import WriteMode

# One authenticated client, reused by the examples below.
dbx = dropbox.Dropbox(os.environ["DROPBOX_ACCESS_TOKEN"])
try:
    dbx.users_get_current_account()  # a cheap call that fails fast on a bad or expired token
except AuthError as err:
    raise RuntimeError("DROPBOX_ACCESS_TOKEN was rejected; create a new token") from err

local_file = Path(tempfile.mkdtemp(prefix="dropbox_demo_")) / "notes.txt"
local_file.write_text("Hello from a notebook\n")
remote_path = "/example_folder/notes.txt"

try:
    # WriteMode("overwrite") replaces a file that already exists at remote_path instead of failing.
    meta = dbx.files_upload(local_file.read_bytes(), remote_path, mode=WriteMode("overwrite"))
    print(f"uploaded {meta.size} bytes to {meta.path_display}")
except ApiError as err:
    # err.error describes the failure, for example a path conflict or a full account.
    print("upload failed:", err.error)

## Example 2

After you've authenticated and setup your client, see contents of your root directory in dropbox

In [ ]:
# Reuses the authenticated `dbx` client created in Example 1
for entry in dbx.files_list_folder(path="").entries:
    print(entry.name)

## Example 3

Using Stone, create a context manager that emits configurable lines before and after an indented block of text.

In [ ]:
from stone.backend import CodeBackend

target_path = "/tmp/ganymede_demo/"

class ExampleBackend(CodeBackend):
    def generate(self, api):
        pass

ex_backend = ExampleBackend(target_path, None)

# Lines emitted inside the block are indented and wrapped in the delimiters ({ and } by default)
with ex_backend.block("void hello()"):
    ex_backend.emit('printf("hello\\n");')

print(ex_backend.output_buffer_to_string())